# 02 · Tiền xử lý dữ liệu

Nhóm 04 · Khai phá dữ liệu nâng cao. Ứng với Chương 3 mục 2 và Chương 4 mục 3 của báo cáo.

**Đầu vào:** 6 tệp CSV gốc trong `data/raw/`.
**Đầu ra:** 5 tệp Parquet trong `data/processed/`, `data_dictionary.csv`, `fayyad_irani_cut_points.csv`.

Các bước: kiểm tra giá trị thiếu → lấy mẫu 10% người dùng thuộc train (hạt giống 42) → giữ toàn bộ lịch sử prior của họ → kiểm tra tính đại diện → ghép danh mục sản phẩm → rời rạc hóa `days_since_prior_order` theo Fayyad–Irani → xuất tệp và đối chiếu với báo cáo.

Nếu `data/processed/` đã có 5 tệp Parquet được dùng trong báo cáo, notebook ghi kết quả mới vào `data/processed/_kiem_tra_buoc2/` rồi so từng tệp, không ghi đè bản đã dùng.

## 0. Cấu hình

In [1]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Thư mục gốc của repo, dùng chung cho cả năm notebook
REPO_DIR = Path(os.environ.get("REPO_DIR", "/content/drive/MyDrive/Nhom04_ADM/repo" if IN_COLAB else ".."))
RAW_DIR = REPO_DIR / "data" / "raw"
PROC_DIR = REPO_DIR / "data" / "processed"
FIG_DIR = REPO_DIR / "reports" / "figures"
OUT_DIR = REPO_DIR / "outputs"
SEED = 42
print("REPO_DIR =", REPO_DIR.resolve())

sys.path.insert(0, str(REPO_DIR / "src"))
from b5_classification.discretize import mdlp_cut_points   # cùng hàm MDLP dùng ở bước 5


Mounted at /content/drive
REPO_DIR = /content/drive/MyDrive/Nhom04_ADM/repo


In [2]:
CHECKS = []

def doi_chieu(ten, thuc_te, bao_cao, tol=0):
    """Ghi một dòng so sánh giữa số vừa tính và số trong báo cáo."""
    if isinstance(bao_cao, float) or tol:
        khop = abs(float(thuc_te) - float(bao_cao)) <= tol
    else:
        khop = thuc_te == bao_cao
    CHECKS.append({"chi_tieu": ten, "thuc_te": str(thuc_te), "bao_cao": str(bao_cao), "khop": bool(khop)})
    return khop


## 1. Nạp dữ liệu gốc

In [3]:
orders = pd.read_csv(RAW_DIR / "orders.csv", dtype={"order_id": "int32", "user_id": "int32", "order_number": "int16",
                     "order_dow": "int8", "order_hour_of_day": "int8", "days_since_prior_order": "float32"})
op_dtype = {"order_id": "int32", "product_id": "int32", "add_to_cart_order": "int16", "reordered": "int8"}
op_prior = pd.read_csv(RAW_DIR / "order_products__prior.csv", dtype=op_dtype)
op_train = pd.read_csv(RAW_DIR / "order_products__train.csv", dtype=op_dtype)
products = pd.read_csv(RAW_DIR / "products.csv")
aisles = pd.read_csv(RAW_DIR / "aisles.csv")
departments = pd.read_csv(RAW_DIR / "departments.csv")
for ten, t in [("orders", orders), ("order_products__prior", op_prior), ("order_products__train", op_train),
               ("products", products), ("aisles", aisles), ("departments", departments)]:
    print(f"{ten:24s} {t.shape}")


orders                   (3421083, 7)
order_products__prior    (32434489, 4)
order_products__train    (1384617, 4)
products                 (49688, 4)
aisles                   (134, 2)
departments              (21, 2)


## 2. Giá trị thiếu (Bảng 4.7)

In [4]:
miss = {ten: int(t.isna().sum().sum()) for ten, t in
        [("orders", orders), ("op_prior", op_prior), ("op_train", op_train), ("products", products),
         ("aisles", aisles), ("departments", departments)]}
print("Số giá trị thiếu theo bảng:", miss)
d = orders["days_since_prior_order"]
thieu = int(d.isna().sum())
thieu_don_dau = int((d.isna() & (orders["order_number"] == 1)).sum())
print(f"days_since_prior_order thiếu {thieu:,} ({thieu / len(orders):.2%}), trong đó ở đơn đầu tiên: {thieu_don_dau:,}")
doi_chieu("Số giá trị thiếu days_since_prior_order", thieu, 206209)
doi_chieu("Thiếu từ đơn thứ 2 trở đi", thieu - thieu_don_dau, 0)


Số giá trị thiếu theo bảng: {'orders': 206209, 'op_prior': 0, 'op_train': 0, 'products': 0, 'aisles': 0, 'departments': 0}
days_since_prior_order thiếu 206,209 (6.03%), trong đó ở đơn đầu tiên: 206,209


True

## 3. Lấy mẫu 10% người dùng thuộc train (Bảng 4.8)

Báo cáo ghi cách lấy mẫu là 10% người dùng train với `random_state = 42`. Ô dưới thử lần lượt các cách viết thông dụng của câu lệnh này. Nếu repo đã có tệp mẫu được dùng trong báo cáo, cách nào cho đúng danh sách người dùng của tệp đó sẽ được chọn. Nếu không cách nào khớp, notebook dùng danh sách người dùng của tệp mẫu đã chốt và ghi rõ điều đó.

In [5]:
train_orders = orders[orders["eval_set"] == "train"]
train_users = train_orders["user_id"].drop_duplicates()
n_mau = round(0.1 * len(train_users))

def _np_legacy():
    np.random.seed(SEED)
    return pd.Series(np.random.choice(train_users.to_numpy(), size=n_mau, replace=False))

CACH_LAY_MAU = {
    "train_users.sample(frac=0.1, random_state=42)": lambda: train_users.sample(frac=0.1, random_state=SEED),
    "train_orders.sample(frac=0.1, random_state=42).user_id": lambda: train_orders.sample(frac=0.1, random_state=SEED)["user_id"],
    "unique đã sắp xếp rồi sample(frac=0.1)": lambda: pd.Series(np.sort(train_users.unique())).sample(frac=0.1, random_state=SEED),
    "np.random.default_rng(42).choice": lambda: pd.Series(np.random.default_rng(SEED).choice(train_users.to_numpy(), size=n_mau, replace=False)),
    "np.random.seed(42) + np.random.choice": _np_legacy,
}

ref_path = PROC_DIR / "orders_train_sample.parquet"
ref_users = set(pd.read_parquet(ref_path, columns=["user_id"])["user_id"]) if ref_path.exists() else None

cach_chon, users = None, None
for ten, ham in CACH_LAY_MAU.items():
    u = set(ham())
    if ref_users is None or u == ref_users:
        cach_chon, users = ten, u
        break
    print(f"Không khớp tệp mẫu đã chốt: {ten}")

if users is None:
    cach_chon, users = "danh sách người dùng của tệp mẫu đã chốt (orders_train_sample.parquet)", ref_users
print("Cách lấy mẫu được dùng:", cach_chon)
print(f"Người dùng train: {len(train_users):,} | người dùng trong mẫu: {len(users):,}")


Cách lấy mẫu được dùng: train_users.sample(frac=0.1, random_state=42)
Người dùng train: 131,209 | người dùng trong mẫu: 13,121


In [6]:
orders_prior_sample = orders[orders["user_id"].isin(users) & (orders["eval_set"] == "prior")].copy()
orders_train_sample = orders[orders["user_id"].isin(users) & (orders["eval_set"] == "train")].copy()
op_prior_sample = op_prior[op_prior["order_id"].isin(orders_prior_sample["order_id"])].copy()
op_train_sample = op_train[op_train["order_id"].isin(orders_train_sample["order_id"])].copy()

quy_mo = pd.Series({"User toàn bộ train": len(train_users), "User trong mẫu": len(users),
                    "orders_prior_sample": len(orders_prior_sample), "orders_train_sample": len(orders_train_sample),
                    "order_products_prior_sample": len(op_prior_sample),
                    "order_products_train_sample": len(op_train_sample)}, name="Số lượng")
display(quy_mo.to_frame())
for k, v in {"User toàn bộ train": 131209, "User trong mẫu": 13121, "orders_prior_sample": 206657,
             "orders_train_sample": 13121, "order_products_prior_sample": 2066000,
             "order_products_train_sample": 139019}.items():
    doi_chieu(k, int(quy_mo[k]), v)
doi_chieu("Đơn prior không có dòng sản phẩm", len(set(orders_prior_sample["order_id"]) - set(op_prior_sample["order_id"])), 0)


,Số lượng
User toàn bộ train,131209
User trong mẫu,13121
orders_prior_sample,206657
orders_train_sample,13121
order_products_prior_sample,2066000
order_products_train_sample,139019


True

## 4. Kiểm tra tính đại diện của mẫu (Bảng 4.9 và 4.10)

In [7]:
prior_train_users = orders[orders["user_id"].isin(train_users) & (orders["eval_set"] == "prior")]
dow = pd.DataFrame({"Toàn bộ": prior_train_users["order_dow"].value_counts(normalize=True).sort_index() * 100,
                    "Mẫu": orders_prior_sample["order_dow"].value_counts(normalize=True).sort_index() * 100})
dow["Chênh lệch tuyệt đối"] = (dow["Toàn bộ"] - dow["Mẫu"]).abs()
display(dow.round(2))
print(f"Chênh lệch tuyệt đối trung bình: {dow['Chênh lệch tuyệt đối'].mean():.2f} điểm phần trăm")
mo_ta = pd.concat({"order_number - Toàn bộ": prior_train_users["order_number"].describe(),
                   "order_number - Mẫu": orders_prior_sample["order_number"].describe(),
                   "days_since_prior_order - Toàn bộ": prior_train_users["days_since_prior_order"].describe(),
                   "days_since_prior_order - Mẫu": orders_prior_sample["days_since_prior_order"].describe()}, axis=1)
display(mo_ta.round(2))
doi_chieu("Số đơn prior của toàn bộ user train", len(prior_train_users), 2047377)
doi_chieu("Chênh lệch order_dow trung bình (điểm %)", round(dow["Chênh lệch tuyệt đối"].mean(), 2), 0.12, tol=0.005)


,Toàn bộ,Mẫu,Chênh lệch tuyệt đối
order_dow,,,
0,17.40,17.55,0.15
1,17.32,17.07,0.26
2,13.72,13.72,0.00
3,12.80,12.85,0.06
4,12.46,12.44,0.02
5,13.27,13.48,0.22
6,13.03,12.88,0.15


Chênh lệch tuyệt đối trung bình: 0.12 điểm phần trăm


,order_number - Toàn bộ,order_number - Mẫu,days_since_prior_order - Toàn bộ,days_since_prior_order - Mẫu
count,2047377.00,206657.00,1916168.00,193536.00
mean,17.20,17.52,10.70,10.60
std,17.79,18.10,8.96,8.96
min,1.00,1.00,0.00,0.00
25%,5.00,5.00,4.00,4.00
50%,11.00,11.00,7.00,7.00
75%,24.00,24.00,14.00,14.00
max,99.00,99.00,30.00,30.00


True

## 5. Ghép và chuẩn hóa danh mục sản phẩm (Bảng 4.11)

In [8]:
products_full = (products.merge(aisles, on="aisle_id", how="left")
                         .merge(departments, on="department_id", how="left"))
products_full["product_name"] = products_full["product_name"].str.strip()
products_full["aisle"] = products_full["aisle"].str.strip().str.lower()
products_full["department"] = products_full["department"].str.strip().str.lower()
products_full = products_full[["product_id", "product_name", "aisle_id", "aisle", "department_id", "department"]]
products_full = products_full.sort_values("product_id").reset_index(drop=True)
print(products_full.shape, "| thiếu aisle hoặc department:", int(products_full[["aisle", "department"]].isna().sum().sum()))
doi_chieu("Số sản phẩm products_full", len(products_full), 49688)


(49688, 6) | thiếu aisle hoặc department: 0


True

## 6. Rời rạc hóa `days_since_prior_order` theo Fayyad–Irani (Bảng 4.12 và 4.13)

Biến lớp là cột `reordered` của từng dòng sản phẩm prior trong mẫu. Đây là phân tích mô tả, khác với bộ điểm cắt của bước 5 (bước 5 rời rạc hóa theo nhãn mua tiếp và giới hạn 5 khoảng). Ở đây dùng tiêu chuẩn MDL gốc, không giới hạn số khoảng.

In [9]:
line = op_prior_sample.merge(orders_prior_sample[["order_id", "days_since_prior_order"]], on="order_id")
du_dieu_kien = line.dropna(subset=["days_since_prior_order"])
cuts = mdlp_cut_points(du_dieu_kien["days_since_prior_order"].to_numpy(), du_dieu_kien["reordered"].to_numpy(),
                       max_intervals=None)
print(f"{len(du_dieu_kien):,} quan sát đủ điều kiện, {len(cuts)} điểm cắt:", cuts)
doi_chieu("Số quan sát đủ điều kiện", len(du_dieu_kien), 1935005)
doi_chieu("Điểm cắt Fayyad-Irani", cuts, [0.5, 1.5, 2.5, 6.5, 7.5, 8.5, 9.5, 10.5, 15.5, 21.5, 25.5, 29.5])

def nhan_khoang(cuts, max_val=30):
    bien = [-np.inf] + list(cuts) + [np.inf]
    nhan = []
    for a, b in zip(bien[:-1], bien[1:]):
        lo = 0 if a == -np.inf else int(np.ceil(a))
        hi = max_val if b == np.inf else int(np.floor(b))
        nhan.append(f"{lo}+" if b == np.inf else (str(lo) if lo == hi else f"{lo}-{hi}"))
    return bien, nhan

bien, nhan = nhan_khoang(cuts)
orders_prior_sample["days_since_prior_bin"] = pd.cut(orders_prior_sample["days_since_prior_order"], bins=bien,
                                                      labels=nhan, right=False, ordered=True)
line = line.merge(orders_prior_sample[["order_id", "days_since_prior_bin"]], on="order_id")
bang_413 = line.groupby("days_since_prior_bin", observed=True)["reordered"].agg(["size", "sum", "mean"])
bang_413.columns = ["Số quan sát", "Số reordered", "Tỷ lệ reordered"]
display(bang_413)
doi_chieu("Khoảng 30+: số quan sát", int(bang_413.loc["30+", "Số quan sát"]), 195032)
doi_chieu("Khoảng 30+: số reordered", int(bang_413.loc["30+", "Số reordered"]), 87995)


1,935,005 quan sát đủ điều kiện, 12 điểm cắt: [0.5, 1.5, 2.5, 6.5, 7.5, 8.5, 9.5, 10.5, 15.5, 21.5, 25.5, 29.5]


,Số quan sát,Số reordered,Tỷ lệ reordered
days_since_prior_bin,,,
0,28266,20705,0.732505
1,64944,40705,0.626771
2,97944,64110,0.654558
3-6,548149,369538,0.674156
7,220485,152781,0.692931
8,122054,81516,0.667868
9,77439,50358,0.650292
10,62322,40001,0.641844
11-15,260682,162225,0.622310


True

## 7. Xuất tệp và so với bản đã dùng trong báo cáo

In [10]:
TEP = {"orders_prior_sample": orders_prior_sample, "orders_train_sample": orders_train_sample,
       "order_products_prior_sample": op_prior_sample, "order_products_train_sample": op_train_sample,
       "products_full": products_full}
da_co_ban_cu = all((PROC_DIR / f"{k}.parquet").exists() for k in TEP)
DICH = PROC_DIR / "_kiem_tra_buoc2" if da_co_ban_cu else PROC_DIR
DICH.mkdir(parents=True, exist_ok=True)
for k, df in TEP.items():
    df.reset_index(drop=True).to_parquet(DICH / f"{k}.parquet", index=False)
pd.DataFrame({"diem_cat": cuts}).to_csv(DICH / "fayyad_irani_cut_points.csv", index=False)

cot = {}
for k, df in TEP.items():
    for c in df.columns:
        cot.setdefault(c, (k, str(df[c].dtype)))
pd.DataFrame([{"bien": c, "tep_dau_tien": t, "kieu": kieu} for c, (t, kieu) in cot.items()]).to_csv(
    DICH / "data_dictionary.csv", index=False, encoding="utf-8-sig")
doi_chieu("Số biến trong data_dictionary", len(cot), 16)
print("Đã ghi vào", DICH)

if da_co_ban_cu:
    for k, df in TEP.items():
        cu = pd.read_parquet(PROC_DIR / f"{k}.parquet")
        a = df.astype(str).sort_values(list(df.columns)).reset_index(drop=True)
        b = cu[df.columns].astype(str).sort_values(list(df.columns)).reset_index(drop=True)
        doi_chieu(f"{k} giống hệt bản đã dùng", bool(a.equals(b)), True)


Đã ghi vào /content/drive/MyDrive/Nhom04_ADM/repo/data/processed/_kiem_tra_buoc2


## 8. Tổng hợp đối chiếu với báo cáo

In [11]:
bang = pd.DataFrame(CHECKS)
display(bang)
so_lech = int((~bang["khop"]).sum())
print(f"Đối chiếu với báo cáo: {len(bang) - so_lech}/{len(bang)} chỉ tiêu khớp")
assert so_lech == 0, "Có chỉ tiêu lệch với báo cáo, xem bảng trên"


,chi_tieu,thuc_te,bao_cao,khop
0,Số giá trị thiếu days_since_prior_order,206209,206209,True
1,Thiếu từ đơn thứ 2 trở đi,0,0,True
2,User toàn bộ train,131209,131209,True
3,User trong mẫu,13121,13121,True
4,orders_prior_sample,206657,206657,True
5,orders_train_sample,13121,13121,True
6,order_products_prior_sample,2066000,2066000,True
7,order_products_train_sample,139019,139019,True
8,Đơn prior không có dòng sản phẩm,0,0,True
9,Số đơn prior của toàn bộ user train,2047377,2047377,True


Đối chiếu với báo cáo: 22/22 chỉ tiêu khớp
